This next section will be on Modelling.

In [1]:
from pyspark.sql import SparkSession

# Initialise Spark session
spark = (
    SparkSession.builder.appName("Rideshare_Analysis")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)


your 131072x1 screen size is bogus. expect trouble
24/08/25 19:07:21 WARN Utils: Your hostname, LAPTOP-354CCOC2 resolves to a loopback address: 127.0.1.1; using 172.21.14.166 instead (on interface eth0)
24/08/25 19:07:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/25 19:07:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/25 19:07:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# When session restarts, continue progress from here
from pyspark.sql.types import *
from pyspark.ml.linalg import VectorUDT

# Define the schema
schema = StructType([
    StructField("hvfhs_license_num", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("trip_miles", DoubleType(), True),
    StructField("trip_time", LongType(), True),
    StructField("license_vec", VectorUDT(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("hour_of_day", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day_vec", VectorUDT(), True),
    StructField("hour_vec", VectorUDT(), True),
    StructField("PULocation_vec", VectorUDT(), True),
    StructField("trip_miles_standardised", DoubleType(), True),
    StructField("earnings_per_hour", DoubleType(), True),
    StructField("feelslike", DoubleType(), True),
    StructField("precip", DoubleType(), True),
    StructField("preciptype", StringType(), True),
    StructField("feelslike_standardised", DoubleType(), True),
    StructField("precip_standardised", DoubleType(), True),
    StructField("preciptype_index", DoubleType(), True),
    StructField("preciptype_vec", VectorUDT(), True),
    StructField("is_public_holiday", IntegerType(), True)
])

# Load the Parquet file with the defined schema
final_df = spark.read.schema(schema).parquet("../data/curated/chunk_*.parquet")


In [9]:
final_df.describe().show()

24/08/25 19:16:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------------+-----------------+------------------+--------------------+----------+----------------------+--------------------+-------------------+-------------------+
|summary|hvfhs_license_num|      PULocationID|        trip_miles|         trip_time|      day_of_week|       hour_of_day|            month|trip_miles_standardised|earnings_per_hour|         feelslike|              precip|preciptype|feelslike_standardised| precip_standardised|   preciptype_index|  is_public_holiday|
+-------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------------+-----------------+------------------+--------------------+----------+----------------------+--------------------+-------------------+-------------------+
|  count|        106611042|         106611042|   

In [3]:
# Train test split

from pyspark.sql.functions import month

# Filter the DataFrame to create training and test sets
training_df = final_df.filter(month('pickup_datetime').between(5, 10) )  # May to October
test_df = final_df.filter(month('pickup_datetime') == 11)  # November

# Inspect size of sets
print("Training Set: ", training_df.count())
print("Test Set: ", test_df.count())


Training Set:  91490473


Test Set:  15120569


In [4]:
# Feature Selection
selected_columns = [
    'license_vec', 'trip_miles_standardised','day_vec', 'hour_vec', 'PULocation_vec', 
    'feelslike_standardised', 'precip_standardised', 'preciptype_vec', 'is_public_holiday', 'earnings_per_hour'
]

# Feature columns (excludes the target variable)
feature_columns = [
    'license_vec', 'trip_miles_standardised','day_vec', 'hour_vec', 'PULocation_vec', 
    'feelslike_standardised', 'precip_standardised', 'preciptype_vec', 'is_public_holiday'
]

training_df = training_df.select(*selected_columns)
test_df = test_df.select(*selected_columns)

# Preview
training_df.limit(7)


license_vec,trip_miles_standardised,day_vec,hour_vec,PULocation_vec,feelslike_standardised,precip_standardised,preciptype_vec,is_public_holiday,earnings_per_hour
"(1,[],[])",-0.5270626276153315,"(6,[],[])","(23,[17],[1.0])","(260,[152],[1.0])",-0.5020529657866335,0.31966025833655953,"(1,[],[])",0,53.97
"(1,[],[])",-0.6248138157441934,"(6,[],[])","(23,[17],[1.0])","(260,[136],[1.0])",-0.5020529657866335,0.31966025833655953,"(1,[],[])",0,56.59
"(1,[0],[1.0])",-0.5124347776102999,"(6,[],[])","(23,[17],[1.0])","(260,[74],[1.0])",-0.5020529657866335,0.31966025833655953,"(1,[],[])",0,54.83
"(1,[0],[1.0])",1.1453882229599457,"(6,[],[])","(23,[17],[1.0])","(260,[194],[1.0])",-0.5020529657866335,0.31966025833655953,"(1,[],[])",0,81.35
"(1,[0],[1.0])",-0.6656789205201546,"(6,[],[])","(23,[17],[1.0])","(260,[11],[1.0])",-0.5020529657866335,0.31966025833655953,"(1,[],[])",0,48.73
"(1,[0],[1.0])",1.6376269850340246,"(6,[],[])","(23,[17],[1.0])","(260,[55],[1.0])",-0.5020529657866335,0.31966025833655953,"(1,[],[])",0,62.86
"(1,[0],[1.0])",-0.8398199920086258,"(6,[],[])","(23,[17],[1.0])","(260,[70],[1.0])",-0.5020529657866335,0.31966025833655953,"(1,[],[])",0,59.96


In [5]:
# Prepare Data for Linear Regression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# Assemble the features into a feature vector
assembler = VectorAssembler(inputCols=feature_columns, outputCol='features')

# Transform the training and test data
training_data = assembler.transform(training_df).select('features', 'earnings_per_hour')
test_data = assembler.transform(test_df).select('features', 'earnings_per_hour')


In [6]:
import pandas as pd
# Coefficients, Predictions/Result

# Initialise the Linear Regression model and fit on training data
lm = LinearRegression(featuresCol='features', labelCol='earnings_per_hour').fit(training_data)

# Output the coefficients and intercept
print("Coefficients: ", lm.coefficients)
print("Intercept: ", lm.intercept)

# Generate predictions for test data
predictions = lm.transform(test_data)

# Display some of the predictions
predictions.select('features', 'earnings_per_hour', 'prediction').limit(7)

24/08/25 19:09:32 WARN Instrumentation: [d23ed59e] regParam is zero, which might cause numerical instability and overfitting.
24/08/25 19:10:33 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/08/25 19:11:06 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Coefficients:  [3.645418034102357,1.4471040563049964,0.21533701137425598,-0.6121878967823294,-0.9681138199100986,3.093194162643899,-0.6862419583191477,-1.0038927439473495,-12.801431209160638,-11.875639055108145,-14.284080102882745,-9.639151728115305,-11.662788959757616,-7.650515830873253,-15.346641351197869,-14.904492363390988,-5.299654729678516,-14.136994414152232,-10.388423191688377,-11.948340233779334,-13.10753792035926,-13.72626292303478,-12.820337160620344,-13.31564599802174,-8.347116583633916,-4.818084369606243,-6.077678608921232,-4.330207597367523,-2.7362413005571953,-1.2606940231262305,2.495650346301685,-306.2535356942347,-306.0547072467721,-313.593716745487,-317.4258966425904,-310.5511332542237,-311.1732388898368,-307.97987397229355,-313.0171871470882,-312.3202893950603,-312.4742406496125,-311.51378120847767,-312.8259656348176,-317.3275304877285,-313.8700010804134,-313.5643147534478,-314.68979856432856,-314.2250311898887,-312.11788811397207,-313.0768424869622,-316.889445804814

features,earnings_per_hour,prediction
"(295,[0,1,4,18,14...",50.71,54.80954534070128
"(295,[0,1,4,18,95...",54.51,55.00443018529876
"(295,[0,1,4,18,17...",44.39,56.67370586218914
"(295,[0,1,4,18,79...",58.19,66.86284362666919
"(295,[1,4,18,79,2...",64.14,59.01708541022782
"(295,[0,1,4,18,45...",42.98,57.91113553681009
"(295,[0,1,4,18,42...",42.65,58.58900459521942


In [7]:
#Error Analysis -> Evaluate the model performance using RMSE

from pyspark.ml.evaluation import RegressionEvaluator

#Linear Regression
evaluator = RegressionEvaluator(
    labelCol='earnings_per_hour', predictionCol='prediction', metricName='rmse'
)

rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 51.8594


In [8]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml import Pipeline
from pyspark.sql.functions import month

# Filter the DataFrame to create training and test sets
training_df = final_df.filter(month('pickup_datetime').between(5, 10) )  # May to October
test_df = final_df.filter(month('pickup_datetime') == 11)  # November

# Feature columns (excludes the target variable)
feature_columns = [
    'license_vec', 'trip_miles_standardised', 'day_vec', 'hour_vec', 'PULocation_vec', 
    'feelslike_standardised', 'precip_standardised', 'preciptype_vec', 'is_public_holiday'
]

# Assemble the features into a single vector
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Transform the training and test datasets
training_df_transformed = assembler.transform(training_df)
test_df_transformed = assembler.transform(test_df)

# Initialize Decision Tree Regressor
dt = DecisionTreeRegressor(featuresCol="features", labelCol="earnings_per_hour", predictionCol="prediction")

# Train the model
dt_model = dt.fit(training_df_transformed)

# Make predictions on the test data
predictions = dt_model.transform(test_df_transformed)

# Display some of the predictions
predictions.select('features', 'earnings_per_hour', 'prediction').show(5)

ERROR:root:Exception while sending command.                       (0 + 16) / 20]
Traceback (most recent call last):
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o91.fit

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [ ]:



# Convert the predictions to Pandas DataFrame
predictions_rf_pd = predictions_rf.select('prediction', 'earnings_per_hour').toPandas()

# Plot the actual values vs. Random Forest predictions
plt.figure(figsize=(14, 8))
plt.scatter(predictions_rf_pd['earnings_per_hour'], predictions_rf_pd['prediction'], alpha=0.5, label="Random Forest Predictions", color='green')
plt.plot(predictions_rf_pd['earnings_per_hour'], predictions_rf_pd['earnings_per_hour'], color='red', label="Actual Values", linewidth=2)

# Adding labels and title
plt.title('Random Forest Model Predictions vs Actual Values')
plt.xlabel('Actual Earnings per Hour')
plt.ylabel('Predicted Earnings per Hour')
plt.legend()
plt.grid(True)

In [ ]:
#Error Analysis -> Evaluate the model performance using RMSE

from pyspark.ml.evaluation import RegressionEvaluator

#Linear Regression
evaluator = RegressionEvaluator(
    labelCol='earnings_per_hour', predictionCol='prediction', metricName='rmse'
)

rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)


# XGboost
# Evaluate the model
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="earnings_per_hour", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")


# Evaluate the model
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="earnings_per_hour", predictionCol="prediction", metricName="rmse")
rmse_rf = evaluator.evaluate(predictions_rf)

print(f"Root Mean Squared Error (RMSE) on test data = {rmse_rf}")